# Notebook 5 — Evaluación SCE-Net

Implementa las dos métricas estándar del paper (Tan et al., 2019) sobre Polyvore Outfits:
- **Fill-in-the-Blank (FITB)**: dado un outfit parcial, seleccionar el item más compatible entre candidatos
- **Compatibility AUC**: clasificar outfits reales vs outfits aleatorios incompatibles

Se evalúa en los splits **disjoint** y **nondisjoint**.

## 0. Imports y configuración

In [ ]:
import json
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import torch.nn as nn
warnings.filterwarnings('ignore')

# ── Ajusta estas rutas a tu entorno ──────────────────────────────────────────
DATA_ROOT   = Path('polyvore_kaggle/polyvore_outfits')
CKPT_PATH   = Path('checkpoints/best_model_nondisjoint.pth')   # ruta al checkpoint guardado
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ─────────────────────────────────────────────────────────────────────────────

print(f'Device : {DEVICE}')
print(f'Data   : {DATA_ROOT}')

## 1. Cargar modelo y recursos

In [ ]:
outfit_lookup = {}
for split_file in ['nondisjoint/test.json', 'nondisjoint/train.json', 
                   'nondisjoint/valid.json', 'disjoint/test.json',
                   'disjoint/train.json', 'disjoint/valid.json']:
    with open(DATA_ROOT / split_file) as f:
        for outfit in json.load(f):
            set_id = outfit['set_id']
            for item in outfit['items']:
                key = f"{set_id}_{item['index']}"
                outfit_lookup[key] = str(item['item_id'])

print(f'Entradas en lookup: {len(outfit_lookup):,}')
print(f'Ejemplo 217320763_1 → {outfit_lookup.get("217320763_1")}')
print(f'Ejemplo 217320763_4 → {outfit_lookup.get("217320763_4")}')

IMPORTAR LA CLASE SEGUN SI VAMOS A EVALUAR NONDISJOINT O DISJOINT

In [ ]:
from torchvision import models as tv_models

MARGIN    = 0.2

# ── ResNet18 Backbone ───────────────────────────────────────────
class ResNet18Backbone(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        base           = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        self.features  = nn.Sequential(*list(base.children())[:-1])
        self.projection = nn.Linear(512, embed_dim, bias=False)

    # def forward(self, x):
    #     feat = self.features(x).flatten(1)
    #     return F.normalize(self.projection(feat), p=2, dim=-1)


# ── Condition Masks ─────────────────────────────────────────────
class ConditionMasks(nn.Module):
    def __init__(self, m_conditions, embed_dim):
        super().__init__()
        self.masks = nn.Parameter(torch.rand(m_conditions, embed_dim))

    def forward(self, v):
        return v.unsqueeze(1) * self.masks.unsqueeze(0)  # (B, M, D)

    def l1_loss(self):
        return self.masks.abs().sum()


# ── Condition Weight Branch ──────────────────────────────────────
class ConditionWeightBranch(nn.Module):
    def __init__(self, m_conditions, embed_dim, text_dim, hidden_dim=None, temperature=1.0):
    # def __init__(self, m_conditions, embed_dim, text_dim, hidden_dim=None):
        super().__init__()
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim),
            nn.ReLU(),
        )
        hidden = hidden_dim or max(2 * embed_dim, 64)
        self.mlp = nn.Sequential(
            nn.Linear(2 * embed_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, m_conditions),
        )
        self.temperature = temperature

    def forward(self, v_i, v_j, t_i, t_j):
        vt_i   = v_i * self.text_proj(t_i)
        vt_j   = v_j * self.text_proj(t_j)
        logits = self.mlp(torch.cat([vt_i, vt_j], dim=-1))
        return F.softmax(logits / self.temperature, dim=-1)

    # def forward(self, v_i, v_j, t_i, t_j):
    #     vt_i = v_i * self.text_proj(t_i)
    #     vt_j = v_j * self.text_proj(t_j)
    #     return F.softmax(self.mlp(torch.cat([vt_i, vt_j], dim=-1)), dim=-1)


# ── SCE-Net ─────────────────────────────────────────────────────
class SCENet(nn.Module):
    def __init__(self, embed_dim, m_conditions, text_dim, img_size=224, temperature=1.0):
    # def __init__(self, embed_dim, m_conditions, text_dim, img_size=224):
        super().__init__()
        self.backbone      = ResNet18Backbone(embed_dim)
        self.cond_masks    = ConditionMasks(m_conditions, embed_dim)
        self.weight_branch = ConditionWeightBranch(m_conditions, embed_dim, text_dim, temperature=temperature)
        # self.weight_branch = ConditionWeightBranch(m_conditions, embed_dim, text_dim)

    def encode(self, img):
        feat = self.backbone.features(img).flatten(1)  # sin normalizar
        v_raw = self.backbone.projection(feat)          # sin normalizar
        v_norm = F.normalize(v_raw, p=2, dim=-1)        # normalizado
        return v_norm, v_raw                             # devolver ambos

    def get_final_embedding(self, v_i, v_j, t_i, t_j):
        w   = self.weight_branch(v_i, v_j, t_i, t_j)
        O_i = self.cond_masks(v_i)
        O_j = self.cond_masks(v_j)
        w_e = w.unsqueeze(-1)
        E_i = (w_e * O_i).sum(1)
        E_j = (w_e * O_j).sum(1)
        return F.normalize(E_i, p=2, dim=-1), F.normalize(E_j, p=2, dim=-1)
    
    def forward(self, img_a, img_p, img_n, txt_a, txt_p, txt_n):
        v_a, v_a_raw = self.encode(img_a)   # desempaquetar
        v_p, v_p_raw = self.encode(img_p)
        v_n, v_n_raw = self.encode(img_n)
        E_a1, E_p = self.get_final_embedding(v_a, v_p, txt_a, txt_p)
        E_a2, E_n = self.get_final_embedding(v_a, v_n, txt_a, txt_n)
        return (E_a1 + E_a2) / 2.0, E_p, E_n, v_a_raw, v_p_raw, v_n_raw


# ── Loss functions ───────────────────────────────────────────────
def triplet_loss(E_a, E_p, E_n, margin=MARGIN):
    d_pos = (E_a - E_p).norm(dim=-1)
    d_neg = (E_a - E_n).norm(dim=-1)
    return F.relu(d_pos - d_neg + margin).mean()

# def semi_hard_triplet_loss(E_a, E_p, E_n, margin=MARGIN):
#     d_pos = (E_a - E_p).norm(dim=-1)
#     d_neg = (E_a - E_n).norm(dim=-1)
    
#     # Semi-hard: negativos dentro del margen pero más lejos que el positivo
#     semi_hard = (d_neg > d_pos) & (d_neg < d_pos + margin)
    
#     loss = F.relu(d_pos - d_neg + margin)
    
#     if semi_hard.sum() > 0:
#         return loss[semi_hard].mean()
#     return loss.mean()

def l2_reg(v):
    return (v ** 2).sum(dim=-1).mean()

# def vse_loss(v_a, v_p, v_n, t_a, t_p, t_n, margin=0.2):
#     """
#     Visual-Semantic Embedding loss.
#     Cada imagen debe estar más cerca de su propio texto que del texto ajeno.
    
#     v_* : embeddings visuales (B, D)
#     t_* : embeddings de texto proyectados (B, D)
#     """
#     # Proyectar texto al mismo espacio que los embeddings visuales
#     # Normalizar ambos para comparación coseno
#     v_a_n = F.normalize(v_a, p=2, dim=-1)
#     v_p_n = F.normalize(v_p, p=2, dim=-1)
#     v_n_n = F.normalize(v_n, p=2, dim=-1)
#     t_a_n = F.normalize(t_a, p=2, dim=-1)
#     t_p_n = F.normalize(t_p, p=2, dim=-1)
#     t_n_n = F.normalize(t_n, p=2, dim=-1)

#     # Distancia imagen-texto propio vs imagen-texto ajeno
#     # Anchor
#     d_a_ta = (v_a_n - t_a_n).norm(dim=-1)  # imagen_a ↔ texto_a (debería ser pequeña)
#     d_a_tp = (v_a_n - t_p_n).norm(dim=-1)  # imagen_a ↔ texto_p (debería ser grande)
#     d_a_tn = (v_a_n - t_n_n).norm(dim=-1)  # imagen_a ↔ texto_n

#     # Positive
#     d_p_tp = (v_p_n - t_p_n).norm(dim=-1)
#     d_p_ta = (v_p_n - t_a_n).norm(dim=-1)

#     # Negative
#     d_n_tn = (v_n_n - t_n_n).norm(dim=-1)
#     d_n_ta = (v_n_n - t_a_n).norm(dim=-1)

#     l = (F.relu(d_a_ta - d_a_tp + margin) +
#          F.relu(d_a_ta - d_a_tn + margin) +
#          F.relu(d_p_tp - d_p_ta + margin) +
#          F.relu(d_n_tn - d_n_ta + margin))

#     return l.mean()


def sim_loss(v_a, v_p, v_n, margin=0.2):
    """
    Similarity loss.
    Items del mismo outfit (anchor-positive) deben estar más cerca
    entre sí que con el negativo.
    Opera sobre embeddings visuales crudos (antes de máscaras).
    """
    v_a_n = F.normalize(v_a, p=2, dim=-1)
    v_p_n = F.normalize(v_p, p=2, dim=-1)
    v_n_n = F.normalize(v_n, p=2, dim=-1)

    d_pos = (v_a_n - v_p_n).norm(dim=-1)  # anchor ↔ positive (mismo outfit)
    d_neg = (v_a_n - v_n_n).norm(dim=-1)  # anchor ↔ negative (distinto outfit)

    return F.relu(d_pos - d_neg + margin).mean()

def diversity_loss(w, min_entropy=0.5):
    """
    Penaliza solo cuando la entropía cae por debajo de min_entropy.
    No fuerza distribución uniforme — permite especialización.
    """
    entropy = -(w * (w + 1e-8).log()).sum(dim=-1).mean()
    return F.relu(min_entropy - entropy)

def total_loss(model, E_a, E_p, E_n, v_a, v_p, v_n, t_a, t_p, t_n):
    l_t   = triplet_loss(E_a, E_p, E_n)
    l_l1  = model.cond_masks.l1_loss()
    l_l2  = (l2_reg(v_a) + l2_reg(v_p) + l2_reg(v_n)) / 3.0
    l_sim = sim_loss(v_a, v_p, v_n)

    w_ap  = model.weight_branch(v_a, v_p, t_a, t_p)
    w_an  = model.weight_branch(v_a, v_n, t_a, t_n)
    l_div = (diversity_loss(w_ap) + diversity_loss(w_an)) / 2.0

    loss = (l_t
            + LAMBDA_L1 * l_l1
            + LAMBDA_L2 * l_l2
            + LAMBDA_SIM * l_sim
            + LAMBDA_DIV * l_div)

    return loss, l_t, l_l1, l_l2, torch.tensor(0.0), l_sim


print('Clases del modelo definidas OK')

In [ ]:
ckpt   = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
config = ckpt['config']

model = SCENet(
    embed_dim    = config['embed_dim'],
    m_conditions = config['m_conditions'],
    text_dim     = config['text_dim'],
    img_size     = config['img_size'],
    temperature  = 0.1
)
model.load_state_dict(ckpt['model'])
model.to(DEVICE).eval()
epoch      = ckpt['epoch']
valid_trip = ckpt.get('valid_triplet', ckpt.get('valid_loss', 0.0))
print(f"Modelo cargado — época {epoch}, valid_triplet {valid_trip:.4f}")


In [ ]:
# Item index
with open(DATA_ROOT / 'data/processed/item_index_full_nondisjoint.json') as f:
    raw_index = json.load(f)
item_index = {str(k): v for k, v in raw_index.items()}
print(f'Items en índice: {len(item_index):,}')

# Cargar Fisher Vectors
with open(DATA_ROOT / 'data/processed/fisher_item_ids_nondisjoint.json') as f:
    fisher_ids = json.load(f)

fisher_matrix = np.load(DATA_ROOT / 'data/processed/fisher_vectors_nondisjoint.npy')

fv_lookup = {str(item_id): fisher_matrix[i] for i, item_id in enumerate(fisher_ids)}
zero_fv   = np.zeros(config['text_dim'], dtype=np.float32)

print(f'fv_lookup entradas : {len(fv_lookup):,}')
print(f'Dimensión FV       : {fisher_matrix.shape[1]}')

# Transformación de imagen
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Verificación rápida ───────────────────────────────────────────────────────
sample_id = str(fisher_ids[0])
assert sample_id in item_index, f'{sample_id} no está en item_index'
assert sample_id in fv_lookup,  f'{sample_id} no está en fv_lookup'
assert fisher_matrix.shape[1] == config['text_dim'], \
    f'Dimensión FV {fisher_matrix.shape[1]} ≠ text_dim {config["text_dim"]}'
print('✅ Recursos cargados correctamente')

## 2. Funciones auxiliares

In [ ]:
def load_item_tensors(item_id: str):
    item_id = str(item_id)
    meta    = item_index.get(item_id)
    if meta is None:
        return None, None
    try:
        img_path = DATA_ROOT / meta['image_path']  # polyvore_kaggle/polyvore_outfits/images/item_id.jpg
        # print(f'Probando: {img_path} → existe: {img_path.exists()}')  # debug temporal
        img      = Image.open(img_path).convert('RGB')
        img_t    = transform(img).unsqueeze(0).to(DEVICE)
    except Exception as e:
        print(f'Error: {e}')  # debug temporal
        return None, None
    fv    = fv_lookup.get(item_id, zero_fv)
    txt_t = torch.tensor(fv, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    return img_t, txt_t


def get_text_embedding(item_id: str) -> torch.Tensor:
    """
    Recupera el Fisher Vector del item.
    Ajusta esta función según cómo hayas cargado los FV.
    """
    # Opción A — si reutilizas el fv_lookup del dataset:
    # fv = fv_lookup.get(item_id, zero_fv)
    # return torch.tensor(fv, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    # Opción B — placeholder (reemplaza con tu lógica real):
    raise NotImplementedError('Implementa get_text_embedding con tu fv_lookup')


@torch.no_grad()
def get_embedding(item_id: str) -> torch.Tensor | None:
    """Devuelve el embedding final E para un item. Shape: (D,)"""
    img_t, txt_t = load_item_tensors(item_id)
    if img_t is None:
        return None
    v, _ = model.encode(img_t)                           # (1, D)
    return v.squeeze(0)                                  # (D,)


@torch.no_grad()
def get_pair_distance(id_a: str, id_b: str) -> float | None:
    """
    Distancia en el espacio de embeddings finales entre dos items.
    Usa get_final_embedding para aplicar las máscaras de condición.
    """
    img_a, txt_a = load_item_tensors(id_a)
    img_b, txt_b = load_item_tensors(id_b)
    if img_a is None or img_b is None:
        return None

    v_a, _ = model.encode(img_a)
    v_b, _ = model.encode(img_b)
    E_a, E_b = model.get_final_embedding(v_a, v_b, txt_a, txt_b)
    return (E_a - E_b).norm().item()


def outfit_score(item_ids: list[str]) -> float:
    """
    Puntuación de compatibilidad de un outfit como distancia media
    entre todos los pares. Menor = más compatible.
    """
    ids = [str(i) for i in item_ids]
    distances = []
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            d = get_pair_distance(ids[i], ids[j])
            if d is not None:
                distances.append(d)
    return float(np.mean(distances)) if distances else float('inf')

print('Funciones auxiliares listas')

## 3. Fill-in-the-Blank (FITB)

Para cada pregunta hay un outfit de consulta (query items) y 4 candidatos.
El modelo selecciona el candidato con menor distancia media al resto del outfit.

In [ ]:
def evaluate_fitb(fitb_path: Path, max_questions: int = None) -> dict:
    with open(fitb_path) as f:
        questions = json.load(f)
    if max_questions:
        questions = questions[:max_questions]

    correct = 0
    skipped = 0

    for q in tqdm(questions, desc=f'FITB {fitb_path.parent.name}'):
        candidates = q['answers']
        set_id     = str(q['question'][0]).rsplit('_', 1)[0]

        gt_idx = next((i for i, c in enumerate(candidates)
                       if c.rsplit('_', 1)[0] == set_id), None)
        if gt_idx is None:
            skipped += 1
            continue

        cand_ids  = [outfit_lookup.get(c) for c in candidates]
        query_ids = [outfit_lookup.get(str(i)) for i in q['question']]
        query_ids = [i for i in query_ids if i is not None]

        if not query_ids:
            skipped += 1
            continue

        # Puntuar cada candidato — nunca saltar la pregunta entera por un candidato
        scores = []
        for cand_id in cand_ids:
            if cand_id is None:
                scores.append(float('inf'))
                continue
            dists = [get_pair_distance(cand_id, q_id) for q_id in query_ids]
            dists = [d for d in dists if d is not None]
            scores.append(np.mean(dists) if dists else float('inf'))

        # Saltar solo si TODOS los candidatos tienen inf
        if all(s == float('inf') for s in scores):
            skipped += 1
            continue

        if int(np.argmin(scores)) == gt_idx:
            correct += 1

    evaluated = len(questions) - skipped
    print(f'  Outfits procesados: {len(questions)-skipped} / {len(questions)} (skipped={skipped})')
    return {
        'accuracy' : correct / evaluated if evaluated > 0 else 0.0,
        'correct'  : correct,
        'evaluated': evaluated,
        'skipped'  : skipped,
        'total'    : len(questions),
    }

In [ ]:
print('=== Fill-in-the-Blank — Test ===')

# Nondisjoint
fitb_nd = evaluate_fitb(DATA_ROOT / 'nondisjoint/fill_in_blank_test.json')
print(f'Nondisjoint  →  Accuracy: {fitb_nd["accuracy"]*100:.1f}%  '
      f'({fitb_nd["correct"]}/{fitb_nd["evaluated"]})')

# # Disjoint
# fitb_dj = evaluate_fitb(DATA_ROOT / 'disjoint/fill_in_blank_test.json')
# print(f'Disjoint     →  Accuracy: {fitb_dj["accuracy"]*100:.1f}%  '
#       f'({fitb_dj["correct"]}/{fitb_dj["evaluated"]})')

# Referencia del paper (SCE-Net Polyvore Outfits)
print('\nReferencia paper (Polyvore Outfits, nondisjoint): 61.6%')

In [ ]:
with open(DATA_ROOT / 'nondisjoint/fill_in_blank_test.json') as f:
    fitb_data = json.load(f)

q = fitb_data[0]
print('question:', q['question'])
print('answers :', q['answers'])

# set_id extraído
set_id = str(q['question'][0]).rsplit('_', 1)[0]
print(f'\nset_id extraído: {set_id}')

# ¿Algún candidato tiene ese set_id?
for c in q['answers']:
    prefix = c.rsplit('_', 1)[0]
    print(f'  {c} → prefix={prefix} → match={prefix == set_id}')

# ¿Están los IDs en outfit_lookup?
print(f'\nquery en outfit_lookup:')
for i in q['question']:
    print(f'  {i} → {outfit_lookup.get(str(i))}')

print(f'\ncandidatos en outfit_lookup:')
for c in q['answers']:
    print(f'  {c} → {outfit_lookup.get(c)}')

## 4. Compatibility Prediction (AUC)

Para cada outfit real se genera un outfit negativo reemplazando uno de sus items por uno aleatorio de otra categoría.
El AUC mide si el modelo asigna menor distancia media a los outfits reales.

In [ ]:
def load_compatibility_file(txt_path: Path) -> list:
    """
    Lee el fichero compatibility_test.txt.
    Formato: 'label set_id_idx1 set_id_idx2 ...'
    Devuelve lista de (label, [item_ids]).
    """
    samples = []
    with open(txt_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            label    = int(parts[0])
            set_keys = parts[1:]  # formato set_id_index
            item_ids = [outfit_lookup.get(k) for k in set_keys]
            item_ids = [i for i in item_ids if i is not None and i in item_index]
            if len(item_ids) >= 2:
                samples.append((label, item_ids))
    return samples


def evaluate_compatibility(txt_path: Path, max_outfits: int = None) -> dict:
    samples = load_compatibility_file(txt_path)
    if max_outfits:
        samples = samples[:max_outfits]

    scores  = []
    labels  = []
    skipped = 0

    for label, item_ids in tqdm(samples, desc=f'Compat {txt_path.stem}'):
        s = outfit_score(item_ids)
        if not np.isfinite(s):
            skipped += 1
            continue
        scores.append(-s)  # negativo: menor distancia = mayor compatibilidad
        labels.append(label)

    evaluated = len(samples) - skipped
    print(f'  Procesados: {evaluated}/{len(samples)}  skipped={skipped}')
    auc = roc_auc_score(labels, scores)
    return {'auc': auc, 'n_outfits': len(samples), 'n_evaluated': evaluated}

In [ ]:
print('=== Compatibility AUC — Test ===')

compat_nd = evaluate_compatibility(DATA_ROOT / 'nondisjoint/compatibility_test.txt')
print(f'Nondisjoint  →  AUC: {compat_nd["auc"]:.4f}')

# compat_dj = evaluate_compatibility(DATA_ROOT / 'disjoint/compatibility_test.txt')
# print(f'Disjoint     →  AUC: {compat_dj["auc"]:.4f}')

# Referencia del paper (SCE-Net Polyvore Outfits)
print('\nReferencia paper (Polyvore Outfits, nondisjoint): AUC=0.91')

## 5. Resumen de resultados

In [ ]:
print('\n' + '='*55)
print('RESULTADOS FINALES')
print('='*55)
print(f'{"Métrica":<35} {"Nuestro":>8} {"Paper":>8}')
print('-'*55)
print(f'{"FITB Accuracy — Nondisjoint":<35} {fitb_nd["accuracy"]*100:>7.1f}% {61.6:>7.1f}%')
# print(f'{"FITB Accuracy — Disjoint":<35} {fitb_dj["accuracy"]*100:>7.1f}% {"—":>8}')
print(f'{"Compat AUC — Nondisjoint":<35} {compat_nd["auc"]:>8.3f} {0.91:>8.3f}')
# print(f'{"Compat AUC — Disjoint":<35} {compat_dj["auc"]:>8.3f} {"—":>8}')
print('='*55)

## 6. Visualización — Ejemplos cualitativos FITB

In [ ]:
def visualize_fitb_example(fitb_path: Path, question_idx: int = 0):
    with open(fitb_path) as f:
        questions = json.load(f)

    q          = questions[question_idx]
    bp         = q['blank_position']
    candidates = q['answers']

    # Candidato correcto = el que tiene el mismo set_id que question
    set_id = str(q['question'][0]).rsplit('_', 1)[0]
    gt_idx = next((i for i, c in enumerate(candidates)
                   if c.rsplit('_', 1)[0] == set_id), 0)

    # ✅ Resolver set_id_index → item_id real
    cand_ids  = [outfit_lookup.get(c) for c in candidates]
    query_ids = [outfit_lookup.get(str(i)) for i in q['question']]
    query_ids = [i for i in query_ids if i is not None]

    # Calcular scores
    scores = []
    for cand_id in cand_ids:
        if cand_id is None:
            scores.append(float('inf'))
            continue
        dists = [get_pair_distance(cand_id, q_id) for q_id in query_ids]
        dists = [d for d in dists if d is not None]
        scores.append(np.mean(dists) if dists else float('inf'))

    predicted = int(np.argmin(scores))
    n_cols    = max(len(query_ids), len(cand_ids))
    fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 7))

    # Fila 0: outfit de consulta
    for i, item_id in enumerate(query_ids):
        meta = item_index.get(str(item_id), {})
        try:
            img = Image.open(DATA_ROOT / meta['image_path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color='gray')
        axes[0, i].imshow(img)
        axes[0, i].set_title(meta.get('category', '?'), fontsize=9)
        axes[0, i].axis('off')
    for i in range(len(query_ids), n_cols):
        axes[0, i].axis('off')

    # Fila 1: candidatos
    for i, (cand_id, score) in enumerate(zip(cand_ids, scores)):
        meta = item_index.get(str(cand_id), {}) if cand_id else {}
        try:
            img = Image.open(DATA_ROOT / meta['image_path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color='gray')
        axes[1, i].imshow(img)
        color = 'green' if i == gt_idx else ('red' if i == predicted and i != gt_idx else 'black')
        label = f'd={score:.3f}'
        if i == gt_idx:    label += ' ✓GT'
        if i == predicted: label += ' ←pred'
        axes[1, i].set_title(label, fontsize=8, color=color)
        axes[1, i].axis('off')
        for spine in axes[1, i].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
            spine.set_visible(True)

    result = '✅ CORRECTO' if predicted == gt_idx else '❌ INCORRECTO'
    fig.suptitle(f'FITB — Pregunta {question_idx} — {result}', fontsize=12)
    axes[0, 0].set_ylabel('Outfit consulta', fontsize=9)
    axes[1, 0].set_ylabel('Candidatos', fontsize=9)
    plt.tight_layout()
    plt.show()

for idx in [0, 5, 10]:
    visualize_fitb_example(DATA_ROOT / 'nondisjoint/fill_in_blank_test.json', idx)

## 7. Análisis de condiciones aprendidas

Distribución de pesos de condición para distintos tipos de pares.

In [ ]:
@torch.no_grad()
def get_condition_weights(id_a: str, id_b: str) -> np.ndarray | None:
    """Devuelve el vector de pesos w (M,) para un par de items."""
    img_a, txt_a = load_item_tensors(id_a)
    img_b, txt_b = load_item_tensors(id_b)
    if img_a is None or img_b is None:
        return None
    v_a, _ = model.encode(img_a)
    v_b, _ = model.encode(img_b)
    w = model.weight_branch(v_a, v_b, txt_a, txt_b)
    return w.squeeze(0).cpu().numpy()


# Analizar pesos por tipo de par de categorías
with open(DATA_ROOT / 'nondisjoint/fill_in_blank_test.json') as f:
    fitb_data = json.load(f)

cat_weights = {}   # (cat_a, cat_b) → lista de vectores w

for q in tqdm(fitb_data[:200], desc='Analizando pesos'):
    ids = [outfit_lookup.get(str(i)) for i in q['question']]
    ids = [i for i in ids if i is not None]
    for i in range(len(ids)):
        for j in range(i+1, len(ids)):
            w = get_condition_weights(ids[i], ids[j])
            if w is None:
                continue
            cat_a = item_index.get(ids[i], {}).get('category', '?')
            cat_b = item_index.get(ids[j], {}).get('category', '?')
            key = tuple(sorted([cat_a, cat_b]))
            cat_weights.setdefault(key, []).append(w)

# Top 8 pares más frecuentes
top_pairs = sorted(cat_weights.items(), key=lambda x: -len(x[1]))[:8]

M = model.weight_branch.mlp[-1].out_features  # número de condiciones
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for ax, (pair, ws) in zip(axes, top_pairs):
    mean_w = np.mean(ws, axis=0)
    ax.bar(range(1, M+1), mean_w, color='steelblue')
    ax.set_title(f'{pair[0]}\n↔ {pair[1]}\n(n={len(ws)})', fontsize=8)
    ax.set_xlabel('Condición', fontsize=7)
    ax.set_ylabel('Peso medio', fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_xticks(range(1, M+1))

plt.suptitle('Distribución media de pesos de condición por tipo de par', fontsize=11)
plt.tight_layout()
plt.show()

## 8. (Opcional) Evaluación rápida en validación

Útil para comparar checkpoints durante el desarrollo sin esperar al test completo.

In [ ]:
print('=== Validación (nondisjoint) ===')
fitb_val   = evaluate_fitb(DATA_ROOT / 'nondisjoint/fill_in_blank_valid.json',
                            max_questions=500)
compat_val = evaluate_compatibility(DATA_ROOT / 'nondisjoint/valid.json',
                                     max_outfits=300)

print(f'FITB  Accuracy : {fitb_val["accuracy"]*100:.1f}%')
print(f'Compat AUC     : {compat_val["auc"]:.4f}')